In [96]:
from google.cloud import storage
from dotenv import load_dotenv
import os
import pandas as pd
from io import BytesIO
import numpy as np

load_dotenv()

True

In [97]:
GCS_BUCKET = os.getenv("GCS_BUCKET", "").strip()
SILVER_PREFIX = os.getenv("SILVER_PREFIX", "").strip()
SEASON = int(os.getenv("SEASON", ""))
SILVER_FILENAME = os.getenv("SILVER_FILENAME", "").strip()

SILVER_FILEPATH = f"{SILVER_PREFIX}/season={SEASON}/{SILVER_FILENAME}"

POINTS_BY_PLACE = {
        1: 25,
        2: 18,
        3: 15,
        4: 12,
        5: 10,
        6: 8,
        7: 6,
        8: 4,
        9: 2,
        10: 1,
}

In [98]:
client = storage.Client()
blob = client.bucket(GCS_BUCKET).blob(SILVER_FILEPATH)

In [99]:
df = pd.read_parquet(BytesIO(blob.download_as_bytes()))

columns_to_drop = [
    "Stint",
    "PitOutTime",
    "PitInTime",
    "Sector1Time",
    "Sector2Time",
    "Sector3Time",
    "SpeedI1",
    "SpeedI2",
    "SpeedFL",
    "SpeedST",
    "Compound",
    "TyreLife",
    "FreshTyre",
    "TrackStatus",
    "session_type",
    "is_LapTime_not_na",
    "is_pit_out_time_not_na",
    "is_pit_in_time_not_na",
    "is_s1_notna",
    "is_s2_notna",
    "is_s3_notna",
    "is_sector_complete",
    "is_speed_complete",
    "is_green_flag",
    "is_position_not_na",
    "IsPersonalBest"
]

df = df.drop(columns=columns_to_drop, errors="coerce")

df.info()
df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3038 entries, 0 to 3037
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype          
---  ------        --------------  -----          
 0   Time          3038 non-null   timedelta64[ns]
 1   Driver        3038 non-null   object         
 2   DriverNumber  3038 non-null   int64          
 3   LapTime       2990 non-null   timedelta64[ns]
 4   LapNumber     3038 non-null   int64          
 5   Team          3038 non-null   object         
 6   Position      3029 non-null   float64        
 7   season        3038 non-null   int64          
 8   round_number  3038 non-null   int64          
 9   event_name    3038 non-null   object         
dtypes: float64(1), int64(4), object(3), timedelta64[ns](2)
memory usage: 237.5+ KB


,Time,Driver,DriverNumber,LapTime,LapNumber,Team,Position,season,round_number,event_name
0,0 days 01:03:56.437000,NOR,1,0 days 00:01:36.458000,1,McLaren,6.0,2026,1,AUSTRALIAN GRAND PRIX
1,0 days 01:05:23.781000,NOR,1,0 days 00:01:27.344000,2,McLaren,6.0,2026,1,AUSTRALIAN GRAND PRIX
2,0 days 01:06:50.644000,NOR,1,0 days 00:01:26.863000,3,McLaren,7.0,2026,1,AUSTRALIAN GRAND PRIX
3,0 days 01:08:16.501000,NOR,1,0 days 00:01:25.857000,4,McLaren,7.0,2026,1,AUSTRALIAN GRAND PRIX
4,0 days 01:09:42.074000,NOR,1,0 days 00:01:25.573000,5,McLaren,7.0,2026,1,AUSTRALIAN GRAND PRIX
...,...,...,...,...,...,...,...,...,...,...
3033,0 days 01:31:41.771000,BEA,87,0 days 00:01:57.448000,17,Haas F1 Team,19.0,2026,3,JAPANESE GRAND PRIX
3034,0 days 01:33:17.825000,BEA,87,0 days 00:01:36.054000,18,Haas F1 Team,19.0,2026,3,JAPANESE GRAND PRIX
3035,0 days 01:34:53.730000,BEA,87,0 days 00:01:35.905000,19,Haas F1 Team,18.0,2026,3,JAPANESE GRAND PRIX
3036,0 days 01:36:29.334000,BEA,87,0 days 00:01:35.604000,20,Haas F1 Team,18.0,2026,3,JAPANESE GRAND PRIX


In [100]:
def insert_driver(overall, one):
    keys = ["Driver", "DriverNumber", "Team"]

    all_drivers = (
        overall[keys]
        .drop_duplicates(subset=keys)
        .assign(
            Driver = lambda d: d["Driver"].astype(str).str.strip(),
            DriverNumber = lambda d: d["DriverNumber"].astype(int),
        ).sort_values("DriverNumber", ignore_index=True, ascending=True)
    )

    drivers = (
        one[keys]
        .drop_duplicates(subset=keys)
        .assign(
            Driver = lambda d: d["Driver"].astype(str).str.strip(),
            DriverNumber = lambda d: d["DriverNumber"].astype(int),
        ).sort_values("DriverNumber", ignore_index=True, ascending=True)
    )

    miss = all_drivers.merge(drivers, on=keys, how="left", indicator=True)
    miss = miss.loc[miss["_merge"] == "left_only", keys].reset_index(drop=True)
    
    if miss.empty:
        return one.copy()
    
    stub = pd.DataFrame({c: np.nan for c in one.columns}, index=miss.index)
    stub["DriverNumber"] = miss["DriverNumber"].values
    stub["Driver"] = miss["Driver"].values
    stub["Team"] = miss["Team"].values
    if "season" in one.columns:
        stub["season"] = one["season"].iloc[0]
    if "round_number" in one.columns:
        stub["round_number"] = one["round_number"].iloc[0]
    if "event_name" in one.columns:
        stub["event_name"] = one["event_name"].iloc[0]
    
    return pd.concat([one.reset_index(drop=True), stub], ignore_index=True)

def assign_points(position):
    points = position.map(POINTS_BY_PLACE)
    points = points.fillna(0)
    return points

**RACE 1**

In [104]:
for r in round:
    race1_base = df[(df["round_number"]) == r].copy()

    pb = race1_base.sort_values(
        by=["DriverNumber", "LapTime"],
        ascending=[True, True]
    )

    pb = pb.drop_duplicates(subset="DriverNumber", keep="first")
    pb = pb[["DriverNumber", "LapTime"]].rename(columns={"LapTime": "PersonalBest"})

    race1 = race1_base.drop(columns="LapTime")
    race1 = race1.sort_values(
        by=["DriverNumber", "Time"],
        ascending=[True, False]
    )
    race1 = race1.drop_duplicates(subset="DriverNumber", keep="first")

    overall_roster = df
    race1 = insert_driver(overall_roster, race1)

    race1["Position"] = race1["Position"].fillna(100)

    race1["Points"] = 0
    race1["Points"] = assign_points(race1["Position"])

    race1["Position"] = pd.to_numeric(race1["Position"], errors="coerce").astype(int)
    race1 = race1.sort_values(
        by=["Position", "Time"],
        ascending=[True, False]
    )
    race1 = race1.merge(pb, on=["DriverNumber"], how="left")

    pos = race1["Position"]
    lap = race1["PersonalBest"]

    race1.loc[(pos == 100) & lap.isna(), "Position"] = "DNS"
    race1.loc[(pos == 100) & lap.notna(), "Position"] = "DNF"

    race1["Position"] = race1["Position"].astype(str)
    buf = BytesIO()
    race1.to_parquet(buf, index=False)
    buf.seek(0)

    bucket = client.bucket(GCS_BUCKET)
    key = f"gold/season={SEASON}/round={r:02d}/report.parquet"
    bucket.blob(key).upload_from_file(buf, content_type="application/octet-stream")


C:\Users\USER\AppData\Local\Temp\ipykernel_32168\1336846875.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([one.reset_index(drop=True), stub], ignore_index=True)
C:\Users\USER\AppData\Local\Temp\ipykernel_32168\165311002.py:37: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'DNS' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  race1.loc[(pos == 100) & lap.isna(), "Position"] = "DNS"
C:\Users\USER\AppData\Local\Temp\ipykernel_32168\165311002.py:37: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'DNS' has dtype incompati